# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. We follow reproducible steps for metadata inspection, data extraction, and basic EDA based on the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as specified in the Croissant schema.

**Note:** All references use the `@id` of each entity, as required for unambiguous reference.

<details>
  <summary>Show dataset structure via Croissant @ids (click to expand)</summary>

The main record set in this dataset (from FAIR² metadata) is:

- `cr:recordSet/protein_table`: This is the record set containing per-protein quantitative and annotation fields.

**Example field `@id`s and columns inside `cr:recordSet/protein_table`:**

| Field Description                             | Field `@id`                        | Column `@id`                   |
|:----------------------------------------------|:-----------------------------------|:-------------------------------|
| UniProt Accession number                      | cr:field/accession                 | cr:column/accession            |
| Protein description/name                      | cr:field/description               | cr:column/description          |
| Sequence coverage (%)                         | cr:field/coverage_pct              | cr:column/coverage_pct         |
| Peptide count                                 | cr:field/peptide_count             | cr:column/peptide_count        |
| Molecular weight (kDa)                        | cr:field/mw                        | cr:column/mw                   |
| Calculated isoelectric point (pI)             | cr:field/pi                        | cr:column/pi                   |
| Normalized abundance (sample 1)               | cr:field/abundance_sample1         | cr:column/abundance_sample1    |
| Normalized abundance (sample 2)               | cr:field/abundance_sample2         | cr:column/abundance_sample2    |
| ...                                           | ...                                | ...                            |

</details>

In [ ]:
# List all available record set @ids
print("Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}")

# For this dataset, print fields and columns for primary record set:
record_set_id = 'cr:recordSet/protein_table'  # replace with actual @id if needed
record_sets = {r['@id']: r for r in dataset.metadata.record_sets}

if record_set_id in record_sets:
    rs_obj = record_sets[record_set_id]
    print(f"\nFields and columns in record set '{record_set_id}':")
    for field in rs_obj['field']:
        field_id = field['@id']
        field_desc = field.get('description', '')
        # Each field typically has a column
        col = field.get('column')
        print(f"- Field: {field_id} | Column: {col['@id'] if isinstance(col, dict) else col} | Desc: {field_desc}")

## 3. Data Extraction
Load data from the primary record set (`cr:recordSet/protein_table`, using its `@id`) into a DataFrame for analysis.

In [ ]:
# The primary record set @id for the protein table
record_sets = ['cr:recordSet/protein_table']
dataframes = {}

for record_set in record_sets:
    # The .records() generator yields dicts; columns are from field/column @ids
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

print("Columns (by column @id):", dataframes['cr:recordSet/protein_table'].columns.tolist())
# Show a few entries
dataframes['cr:recordSet/protein_table'].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by a numeric field, normalizing, and grouping. All fields are referenced by their `@id` **(see previous overview cell for reference)**.

### Example workflow:
- Filter records for proteins with coverage percentage > 10.
- Normalize the molecular weight (`cr:column/mw`).
- Group proteins by the number of peptides (`cr:column/peptide_count`, if categorical).


In [ ]:
# Specify the record set, field, and group column @ids
record_set_id = 'cr:recordSet/protein_table'
coverage_col = 'cr:column/coverage_pct'        # numeric field
mw_col = 'cr:column/mw'                        # molecular weight
peptide_count_col = 'cr:column/peptide_count'  # can be used for grouping

# Filter for sequence coverage > 10%
df = dataframes[record_set_id]
if coverage_col in df.columns:
    filtered_df = df[pd.to_numeric(df[coverage_col], errors='coerce') > 10]
    print(f"Filtered records with {coverage_col} > 10 (coverage > 10%): {len(filtered_df)} entries")
else:
    print(f"Field {coverage_col} not in DataFrame columns.")

# Normalize molecular weight
if mw_col in filtered_df.columns:
    filtered_df[mw_col] = pd.to_numeric(filtered_df[mw_col], errors='coerce')
    mean_mw = filtered_df[mw_col].mean()
    std_mw = filtered_df[mw_col].std()
    filtered_df[f'{mw_col}_normalized'] = (filtered_df[mw_col] - mean_mw) / std_mw
    print(filtered_df[[mw_col, f'{mw_col}_normalized']].head())
else:
    print(f"Molecular weight column ({mw_col}) missing!")

# Group by peptide count if available
if peptide_count_col in filtered_df.columns:
    # Let's bin peptide counts for grouping
    filtered_df[peptide_count_col] = pd.to_numeric(filtered_df[peptide_count_col], errors='coerce')
    peptide_bins = pd.cut(filtered_df[peptide_count_col], bins=[0,2,5,10,50,100,filtered_df[peptide_count_col].max()],
                          labels=['1-2','3-5','6-10','11-50','51-100','100+'])
    grouped_df = filtered_df.groupby(peptide_bins).agg({mw_col:'mean', f'{mw_col}_normalized':'mean'})
    print("\nAvg molecular weight by peptide count bin:")
    print(grouped_df)
else:
    print(f"Peptide count column ({peptide_count_col}) missing!")

## 5. Visualization
Visualize the distribution of protein molecular weights and the relation to peptide counts in the filtered protein table.

We use matplotlib and seaborn for plotting (install if needed):

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of molecular weights (filtered)
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[mw_col].dropna(), bins=30, kde=True)
plt.title('Distribution of Protein Molecular Weight (kDa)')
plt.xlabel('Molecular Weight (kDa)')
plt.ylabel('Number of Proteins')
plt.show()

# Scatter plot: Peptide count vs. Molecular weight
plt.figure(figsize=(8,5))
sns.scatterplot(data=filtered_df, x=peptide_count_col, y=mw_col)
plt.title('Peptide Count vs. Protein Molecular Weight')
plt.xlabel('Number of Peptides')
plt.ylabel('Molecular Weight (kDa)')
plt.show()

## 6. Conclusion

- We loaded and inspected the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant` with precise `@id` references.
- The dataset organizes protein-level quantitative and annotation data in a central record set (`cr:recordSet/protein_table`), programmatically extracted and analyzed in this notebook.
- Filtering and normalization revealed meaningful subsets, and visualizations provided insights into the molecular weight and peptide count distributions in the dataset.

**This workflow can be expanded for deeper analysis or integrated into bioinformatics applications requiring machine-readable, FAIR-compliant data loading.**